In [ ]:
# =========================================================
# UK HOUSING AFFORDABILITY - CLEAN DATA PIPELINE
# =========================================================

import os
import pandas as pd
from google.colab import drive

# =========================================================
# 1. MOUNT DRIVE
# =========================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# 2. DEFINE PROJECT PATHS
# =========================================================

project_path = "/content/drive/MyDrive/DataAnalytics/UKHousingAffordabilityAnalysis"

data_path = os.path.join(project_path, "Data")
raw_data_path = os.path.join(data_path, "Raw")
interim_data_path = os.path.join(data_path, "Interim")
processed_data_path = os.path.join(data_path, "Processed")

outputs_path = os.path.join(project_path, "Outputs")
metrics_path = os.path.join(outputs_path, "Metrics")
tables_path = os.path.join(outputs_path, "Tables")
figures_path = os.path.join(outputs_path, "Figures")
tableau_path = os.path.join(outputs_path, "Tableau")

for path in [
    raw_data_path,
    interim_data_path,
    processed_data_path,
    metrics_path,
    tables_path,
    figures_path,
    tableau_path,
]:
    os.makedirs(path, exist_ok=True)



In [ ]:
# =========================================================
# 3. FILE PATHS
# =========================================================

earnings_workbook = os.path.join(
    raw_data_path,
    "aff2ratioofhousepricetoresidencebasedearnings.xlsx"
)

income_workbook = os.path.join(
    raw_data_path,
    "housingpurchaseaffordabilityratioslocalauthorityareasinenglandandwales.xlsx"
)

print("Earnings workbook exists:", os.path.exists(earnings_workbook))
print("Income workbook exists:", os.path.exists(income_workbook))



Earnings workbook exists: True
Income workbook exists: True


In [ ]:
# =========================================================
# 4. HELPER FUNCTION - LOAD ONS SHEET
# =========================================================

def load_ons_sheet(file_path, sheet_name, header_keyword):
    preview = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(15):
        row_values = preview.iloc[i].astype(str).str.lower().tolist()
        if header_keyword.lower() in row_values:
            header_row = i
            break

    if header_row is None:
        raise ValueError(f"Header row not found in sheet: {sheet_name}")

    df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
    )

    df = df.dropna(how="all").reset_index(drop=True)
    return df



In [ ]:
# =========================================================
# 5. LOAD SHEETS
# =========================================================

table_5a = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5a",
    header_keyword="Local authority code"
)

table_5b = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5b",
    header_keyword="Local authority code"
)

table_5c = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5c",
    header_keyword="Local authority code"
)

table_1 = load_ons_sheet(
    file_path=income_workbook,
    sheet_name="1",
    header_keyword="Region code"
)

print("Table 5a shape:", table_5a.shape)
print("Table 5b shape:", table_5b.shape)
print("Table 5c shape:", table_5c.shape)
print("Table 1 shape:", table_1.shape)

# =========================================================
# 6. REMOVE UNNECESSARY COLUMN
# =========================================================

table_5c = table_5c.drop(columns=["5-year average"], errors="ignore")



Table 5a shape: (318, 27)
Table 5b shape: (318, 27)
Table 5c shape: (318, 28)
Table 1 shape: (317, 30)


In [ ]:
# =========================================================
# 7. RESHAPE DATASET 1: WIDE -> LONG
# =========================================================

id_vars_dataset1 = [
    "Country/Region code",
    "Country/Region name",
    "Local authority code",
    "Local authority name"
]

price_long = table_5a.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Median_House_Price"
)

earnings_long = table_5b.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Median_Earnings"
)

ratio_long = table_5c.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Affordability_Ratio_Earnings"
)

# Clean year columns
price_long["Year"] = price_long["Year"].str.extract(r"(\d{4})").astype(int)
earnings_long["Year"] = earnings_long["Year"].str.extract(r"(\d{4})").astype(int)
ratio_long["Year"] = ratio_long["Year"].str.extract(r"(\d{4})").astype(int)



In [ ]:
# =========================================================
# 8. MERGE DATASET 1 TABLES
# =========================================================

dataset1 = (
    price_long
    .merge(
        earnings_long,
        on=id_vars_dataset1 + ["Year"],
        how="inner"
    )
    .merge(
        ratio_long,
        on=id_vars_dataset1 + ["Year"],
        how="inner"
    )
)

print("Dataset1 shape:", dataset1.shape)



Dataset1 shape: (7314, 8)


In [ ]:
# =========================================================
# 9. RESHAPE DATASET 2: WIDE -> LONG
# =========================================================

income_long = table_1.melt(
    id_vars=[
        "Region code",
        "Region name",
        "Local authority code",
        "Local authority name"
    ],
    var_name="Year",
    value_name="Affordability_Ratio_Income"
)

# Convert financial year ending to standard year
# Example: 1998/99 -> 1999, 2023/24 -> 2024
income_long["Year"] = income_long["Year"].str[-2:].astype(int) + 2000



In [ ]:
# =========================================================
# 10. MERGE BOTH DATASETS
# =========================================================

final_df = dataset1.merge(
    income_long,
    on=["Local authority code", "Local authority name", "Year"],
    how="left"
)

print("Merged dataset shape:", final_df.shape)


Merged dataset shape: (7314, 11)


In [ ]:
# =========================================================
# 11. DATA TYPE CLEANING
# =========================================================

numeric_cols = [
    "Median_House_Price",
    "Median_Earnings",
    "Affordability_Ratio_Earnings",
    "Affordability_Ratio_Income"
]

for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

final_df["Year"] = pd.to_numeric(final_df["Year"], errors="coerce").astype(int)



In [ ]:
# =========================================================
# 12. FILL REGION VALUES BEFORE DROPPING DUPLICATES
# =========================================================

if "Region name" in final_df.columns and "Country/Region name" in final_df.columns:
    final_df["Region name"] = final_df["Region name"].fillna(final_df["Country/Region name"])

if "Region code" in final_df.columns and "Country/Region code" in final_df.columns:
    final_df["Region code"] = final_df["Region code"].fillna(final_df["Country/Region code"])



In [ ]:
# =========================================================
# 13. CREATE KPI
# =========================================================

final_df["Affordability_Gap"] = (
    final_df["Affordability_Ratio_Income"] -
    final_df["Affordability_Ratio_Earnings"]
)



In [ ]:
# =========================================================
# 14. QUALITY CHECKS BEFORE DROPPING
# =========================================================

final_df_raw = final_df.copy()
initial_rows = len(final_df)

rows_missing_earnings = final_df["Median_Earnings"].isna().sum()
rows_missing_ratio_earnings = final_df["Affordability_Ratio_Earnings"].isna().sum()
rows_missing_ratio_income = final_df["Affordability_Ratio_Income"].isna().sum()

rows_missing_core = final_df[
    final_df["Median_Earnings"].isna() |
    final_df["Affordability_Ratio_Earnings"].isna()
].shape[0]



In [ ]:
# =========================================================
# 15. DROP ROWS MISSING CORE FIELDS
# =========================================================

final_df = final_df.dropna(
    subset=["Median_Earnings", "Affordability_Ratio_Earnings"]
).copy()

# =========================================================
# 16. DROP DUPLICATE REGION COLUMNS
# =========================================================

final_df = final_df.drop(
    columns=["Country/Region code", "Country/Region name"],
    errors="ignore"
)

final_df = final_df.reset_index(drop=True)



In [ ]:
# =========================================================
# 17. FINAL AUDIT SUMMARY
# =========================================================

final_rows = len(final_df)
rows_removed = initial_rows - final_rows

missing_audit = pd.DataFrame({
    "Metric": [
        "Initial rows",
        "Rows missing Median_Earnings",
        "Rows missing Affordability_Ratio_Earnings",
        "Rows missing Affordability_Ratio_Income",
        "Rows missing core fields dropped",
        "Rows removed total",
        "Final rows"
    ],
    "Value": [
        initial_rows,
        rows_missing_earnings,
        rows_missing_ratio_earnings,
        rows_missing_ratio_income,
        rows_missing_core,
        rows_removed,
        final_rows
    ]
})

print("\nMissing Values Audit Summary:")
print(missing_audit)

print("\nFinal missing values:")
print(final_df.isna().sum())

print("\nDuplicate rows:", final_df.duplicated().sum())
print("Year range:", final_df["Year"].min(), "to", final_df["Year"].max())
print("Unique Local Authorities:", final_df["Local authority name"].nunique())
print("Unique Regions:", final_df["Region name"].nunique())

print("\nFinal dataset shape:", final_df.shape)
print("\nFinal dataset preview:")
print(final_df.head())

print("\nSummary statistics:")
print(final_df.describe())




Missing Values Audit Summary:
                                      Metric  Value
0                               Initial rows   7314
1               Rows missing Median_Earnings     59
2  Rows missing Affordability_Ratio_Earnings     59
3    Rows missing Affordability_Ratio_Income     71
4           Rows missing core fields dropped     59
5                         Rows removed total     59
6                                 Final rows   7255

Final missing values:
Local authority code             0
Local authority name             0
Year                             0
Median_House_Price               0
Median_Earnings                  0
Affordability_Ratio_Earnings     0
Region code                      0
Region name                      0
Affordability_Ratio_Income      29
Affordability_Gap               29
dtype: int64

Duplicate rows: 0
Year range: 2002 to 2024
Unique Local Authorities: 318
Unique Regions: 10

Final dataset shape: (7255, 10)

Final dataset preview:
  Local authority

In [ ]:
# =========================================================
# 18. SAVE OUTPUTS
# =========================================================

final_df.to_csv(
    os.path.join(processed_data_path, "UKHousingAffordabilityDataset.csv"),
    index=False
)

missing_audit.to_csv(
    os.path.join(tables_path, "MissingValuesAuditSummary.csv"),
    index=False
)